# 🛡️ FraudShield v3 — Inference Server

This notebook:
1. Downloads your trained model from Google Drive
2. Loads the full hybrid prediction pipeline (transformer + URL analyzer + rule engine)
3. Starts a FastAPI server
4. Exposes a public HTTPS endpoint via Cloudflare Tunnel
5. Shows you the endpoint URL with a **copy button**

**Runtime:** Use GPU (Runtime → Change runtime type → T4 GPU)

**Keep this tab open** — the server runs as long as the Colab session is alive (~12h on free tier).

## Step 1 — Install Dependencies

In [ ]:
!pip install -q transformers fastapi uvicorn pyngrok nest-asyncio gdown
print("✅ Dependencies installed")

## Step 2 — Download Model from Google Drive

**Update `FOLDER_ID`** with the ID from your Drive link.

Your link: `https://drive.google.com/drive/folders/1C_P7bjhV2n9y06PL78LqwUyoSrUq9BBI`
So your folder ID is already filled in below.

In [ ]:
import os, glob
import gdown

# This folder ID points DIRECTLY to fraudshield_v3_model/
# Files inside: config.json, model.safetensors, tokenizer_config.json,
#               tokenizer.json, training_args.bin
FOLDER_ID = "1C_P7bjhV2n9y06PL78LqwUyoSrUq9BBI"
MODEL_DIR = "/content/fraudshield_v3_model"

if not os.path.exists(MODEL_DIR) or not os.path.exists(os.path.join(MODEL_DIR, 'config.json')):
    print('📥 Downloading model from Google Drive...')
    os.makedirs(MODEL_DIR, exist_ok=True)
    gdown.download_folder(
        id=FOLDER_ID,
        output=MODEL_DIR,
        quiet=False,
        use_cookies=False
    )
    # gdown sometimes wraps in a subfolder — flatten if needed
    nested = os.path.join(MODEL_DIR, 'fraudshield_v3_model')
    if os.path.isdir(nested):
        import shutil
        for f in os.listdir(nested):
            shutil.move(os.path.join(nested, f), MODEL_DIR)
        os.rmdir(nested)
        print('📁 Flattened nested subfolder')
    print('✅ Download complete')
else:
    print('✅ Model already present, skipping download')

print(f'\n📂 Model directory: {MODEL_DIR}')
print('Files:')
for f in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(os.path.join(MODEL_DIR, f))
    print(f'  {f:45s}  {size/1e6:.2f} MB')

assert os.path.exists(os.path.join(MODEL_DIR, 'config.json')), \
    '❌ config.json missing — check Drive sharing (must be Anyone with link can view)'
assert os.path.exists(os.path.join(MODEL_DIR, 'model.safetensors')), \
    '❌ model.safetensors missing'
print('\n✅ All required files present')


## Step 3 — Imports & Setup

In [ ]:
import re
import torch
import numpy as np
import warnings
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## Step 4 — Class Labels

In [ ]:
CLASS_NAMES = [
    'benign',
    'kyc_scam',
    'impersonation',
    'phishing_link',
    'fake_payment_portal',
    'account_block_scam',
    'vishing',
    'delivery_scam',
    'social_engineering',
    'investment_scam',
]

NUM_LABELS = len(CLASS_NAMES)
label2id   = {c: i for i, c in enumerate(CLASS_NAMES)}
id2label   = {i: c for i, c in enumerate(CLASS_NAMES)}
MAX_LEN    = 128

print(f"✅ {NUM_LABELS} classes loaded")

## Step 5 — URL Analyzer

In [ ]:
SAFE_DOMAINS = {
    'amazon.in', 'flipkart.com', 'hdfcbank.com', 'sbi.co.in',
    'incometax.gov.in', 'uidai.gov.in', 'npci.org.in', 'phonepe.com',
    'paytm.com', 'google.com', 'irctc.co.in', 'icicibank.com',
    'axisbank.com', 'kotak.com',
}

BRAND_NAMES = [
    'amazon', 'flipkart', 'hdfc', 'sbi', 'icici', 'axis', 'kotak',
    'phonepe', 'paytm', 'google', 'irctc', 'uidai', 'npci', 'fedex',
    'dhl', 'lic', 'epfo', 'trai', 'microsoft', 'apple', 'netflix',
]

def analyze_url(message):
    """Extract and analyse any URLs present in the message."""
    urls = re.findall(r'https?://[^\s]+|www\.[^\s]+', message, re.IGNORECASE)
    if not urls:
        return {'has_url': False, 'urls': [], 'risk_score': 0.0, 'flags': []}

    flags = []
    risk  = 0.0

    for url in urls:
        u = url.lower()

        # shorteners
        if re.search(r'bit\.ly|tinyurl|goo\.gl|ow\.ly|is\.gd|cutt\.ly|rb\.gy', u):
            flags.append('url_shortener'); risk = max(risk, 0.65)

        # raw IP
        if re.search(r'https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', u):
            flags.append('raw_ip_address'); risk = max(risk, 0.85)

        # suspicious keywords in path/domain
        sus_kw = ['-kyc-', '-secure-', '-verify-', '-update-', '-login-',
                  '-refund-', 'securelogin', 'taxportal', 'gov-', '-gov.',
                  'reschedule', 'custom-clearance', 'kyc-update']
        for kw in sus_kw:
            if kw in u:
                flags.append(f'suspicious_keyword:{kw.strip("-")}'); risk = max(risk, 0.75)

        # brand in non-brand domain (spoofing)
        domain_match = re.search(r'https?://([^/]+)', u)
        if domain_match:
            domain = domain_match.group(1)
            if not any(domain.endswith(sd) for sd in SAFE_DOMAINS):
                for brand in BRAND_NAMES:
                    if brand in domain:
                        flags.append(f'brand_spoofing:{brand}'); risk = max(risk, 0.90)

        # suspicious TLDs
        if re.search(r'\.(xyz|top|club|work|online|site|tk|ml|ga|cf|gq)(\.|/|$)', u):
            flags.append('suspicious_tld'); risk = max(risk, 0.60)

        # .in subdomains impersonating gov
        if re.search(r'(gov|rbi|trai|uidai|npci|income.tax)\.(?!gov\.in|uidai\.gov)', u):
            flags.append('gov_impersonation'); risk = max(risk, 0.92)

    return {
        'has_url':    True,
        'urls':       urls,
        'risk_score': round(risk, 3),
        'flags':      list(set(flags)),
    }

print("✅ URL analyzer ready")

## Step 6 — Rule Engine

In [ ]:
def rule_engine(text):
    """28+ pattern families for Indian + global scam detection."""
    if not isinstance(text, str) or not text.strip():
        return None, 0.0, 'empty_input'

    t = text.lower()
    has_url = bool(re.search(r'https?://|www\.', t))
    has_sus_url = bool(re.search(
        r'(bit\.ly|tinyurl|goo\.gl|ow\.ly|is\.gd|cutt\.ly|rb\.gy|t\.co'
        r'|tiny\.one|shorturl|hxxp|\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}'
        r'|-kyc-|-secure-|-verify-|-update-|-login-|-refund-'
        r'|securelogin|taxportal|gov-[a-z]|[a-z]-gov\.|reschedule)',
        t
    ))

    RULES = [
        # KYC scams
        (r'kyc.{0,20}(expire|expir|updat|verif|complet|pending|suspend)', 'kyc_scam', 0.93),
        (r'(update|complete|verify).{0,20}kyc', 'kyc_scam', 0.91),
        (r'account.{0,30}(block|suspend|freez).{0,30}kyc', 'kyc_scam', 0.92),

        # Phishing links
        (r'click.{0,20}(here|link|below).{0,20}(verif|updat|login|confirm)', 'phishing_link', 0.82),
        (r'(login|sign.?in).{0,20}(secure|verif|official)', 'phishing_link', 0.78),

        # Fake payment / UPI scam
        (r'(cashback|reward|prize|won).{0,30}(click|link|claim|collect)', 'fake_payment_portal', 0.85),
        (r'(refund|money back).{0,30}(click|link|form|portal)', 'fake_payment_portal', 0.82),
        (r'upi.{0,20}(reward|cashback|won|prize|gift)', 'fake_payment_portal', 0.87),

        # Account block scam
        (r'(account|card|upi).{0,30}(block|suspend|freez|deactiv)', 'account_block_scam', 0.88),
        (r'(block|suspend).{0,20}(prevent|avoid|update|call)', 'account_block_scam', 0.83),

        # Impersonation (bank/govt)
        (r'(rbi|trai|uidai|npci|income.tax).{0,40}(notice|alert|action|penalt)', 'impersonation', 0.91),
        (r'as per (rbi|trai|uidai|government|ministry)', 'impersonation', 0.88),
        (r'(police|cybercell|court).{0,30}(case|fir|arrest|warrant)', 'impersonation', 0.94),

        # Vishing (fake antivirus / subscription)
        (r'(mcafee|norton|kaspersky|quickheal).{0,40}(renew|subscri|debit|charge)', 'vishing', 0.90),
        (r'rs\.?\s*\d{3,5}.{0,30}(debited|charged|auto.renew).{0,30}(cancel|call)', 'vishing', 0.88),
        (r'call.{0,20}(1800|toll.free|helpline).{0,20}(cancel|refund|stop)', 'vishing', 0.82),

        # Delivery scam
        (r'(package|parcel|shipment).{0,40}(customs|duty|held|clearance).{0,40}(pay|fee|charge)', 'delivery_scam', 0.89),
        (r'(fedex|dhl|bluedart|delhivery).{0,40}(custom|duty|fee|pending)', 'delivery_scam', 0.87),
        (r'delivery.{0,30}(reschedul|missed).{0,30}(click|link|pay)', 'delivery_scam', 0.83),

        # Social engineering (wrong transfer)
        (r'(sent|transfer|paid).{0,20}(wrong|mistake|accident).{0,20}(send back|return|please)', 'social_engineering', 0.88),
        (r'(wrong upi|wrong number).{0,30}(return|send back|refund)', 'social_engineering', 0.87),

        # Investment scam
        (r'(invest|trading|stock).{0,30}(\d{2,3}%.{0,10}(return|profit|gain)|guaranteed)', 'investment_scam', 0.90),
        (r'(whatsapp|telegram).{0,20}(trading|investment|profit|tips)', 'investment_scam', 0.88),
        (r'(daily|weekly|monthly).{0,15}(return|earning|profit).{0,15}(\d{1,3}%|guaranteed)', 'investment_scam', 0.89),

        # OTP sharing
        (r'share.{0,15}otp.{0,15}(agent|executive|officer|team)', 'impersonation', 0.95),
        (r'otp.{0,15}(verify|confirm|complete).{0,15}(kyc|account|process)', 'kyc_scam', 0.90),

        # Lottery / prize
        (r'(congratulation|selected|chosen|lucky winner).{0,40}(prize|reward|gift|cash)', 'fake_payment_portal', 0.87),
        (r'(lottery|lucky draw|contest).{0,30}(won|win|prize|reward)', 'fake_payment_portal', 0.89),
    ]

    for pattern, label, confidence in RULES:
        if re.search(pattern, t):
            if has_sus_url and label in ('phishing_link', 'kyc_scam', 'fake_payment_portal', 'delivery_scam'):
                confidence = min(1.0, confidence + 0.05)
            return label, confidence, f'rule_match:{pattern[:60]}'

    return None, 0.0, 'no_rule_match'

print("✅ Rule engine ready")

## Step 7 — Load Transformer Model

In [ ]:
print(f'Loading model from {MODEL_DIR} ...')
print('Files:', sorted(os.listdir(MODEL_DIR)))

# tokenizer.json (fast tokenizer) + tokenizer_config.json are sufficient
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    use_fast=True
)
print(f'✅ Tokenizer: {type(tokenizer).__name__}')

# model.safetensors is auto-detected by transformers >= 4.26
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    local_files_only=True,
    ignore_mismatched_sizes=True,
)
model.to(device)
model.eval()

print(f'✅ Model loaded on {device}')
total_params = sum(p.numel() for p in model.parameters())
print(f'   Parameters : {total_params/1e6:.1f}M')
print(f'   Weight file: model.safetensors')


## Step 8 — Hybrid Predict Function

In [ ]:
def _run_transformer(message: str) -> np.ndarray:
    inputs = tokenizer(
        message,
        return_tensors='pt',
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN
    ).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0].cpu().numpy()
    return probs


def predict(message: str) -> dict:
    """Full hybrid prediction: transformer + URL analyzer + rule engine in parallel."""
    if not message or not message.strip():
        return {'error': 'Empty message'}

    with ThreadPoolExecutor(max_workers=3) as ex:
        f_url         = ex.submit(analyze_url, message)
        f_rule        = ex.submit(rule_engine, message)
        f_transformer = ex.submit(_run_transformer, message)

    url_result          = f_url.result()
    rule_label, rule_conf, rule_reason = f_rule.result()
    probs               = f_transformer.result()

    # top transformer prediction
    top_idx   = int(np.argmax(probs))
    top_label = CLASS_NAMES[top_idx]
    top_conf  = float(probs[top_idx])

    # --- ensemble fusion ---
    final_label = top_label
    final_conf  = top_conf
    signal_used = 'transformer'

    # rule engine overrides on high confidence
    if rule_label and rule_conf >= 0.88:
        final_label = rule_label
        final_conf  = rule_conf
        signal_used = 'rule_engine'

    # URL risk boosts phishing/kyc predictions
    if url_result['risk_score'] >= 0.85 and top_label in ('phishing_link', 'kyc_scam', 'fake_payment_portal', 'delivery_scam'):
        final_conf  = min(1.0, top_conf + 0.08)
        signal_used = 'transformer+url'

    # high URL risk alone can flag as phishing
    if url_result['risk_score'] >= 0.92 and final_label == 'benign':
        final_label = 'phishing_link'
        final_conf  = url_result['risk_score']
        signal_used = 'url_analyzer'

    # --- verdict ---
    if final_label == 'benign':
        if final_conf >= 0.85:
            verdict = 'SAFE'
        else:
            verdict = 'LIKELY_SAFE'
    elif final_conf >= 0.90:
        verdict = 'SCAM'
    elif final_conf >= 0.70:
        verdict = 'SUSPICIOUS_HIGH_RISK'
    else:
        verdict = 'SUSPICIOUS'

    # per-class probabilities dict
    class_probs = {CLASS_NAMES[i]: round(float(probs[i]), 4) for i in range(NUM_LABELS)}

    return {
        'verdict':       verdict,
        'attack_type':   final_label,
        'confidence':    round(final_conf, 4),
        'signal_used':   signal_used,
        'rule_triggered': rule_reason if rule_label else None,
        'url_analysis':  url_result,
        'class_probabilities': class_probs,
        'top3': sorted(class_probs.items(), key=lambda x: -x[1])[:3],
    }


# quick smoke test
test_msg = "Your KYC is expiring. Update now at sbi-kyc-secure.in or your account will be blocked."
r = predict(test_msg)
print(f"Smoke test → verdict: {r['verdict']}  attack: {r['attack_type']}  conf: {r['confidence']}")
print("✅ Predict function ready")

## Step 9 — FastAPI Server

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import threading
import time

nest_asyncio.apply()

app = FastAPI(
    title="FraudShield v3",
    description="Advanced fraud detection API — 10 attack classes, hybrid ensemble",
    version="3.0.0"
)

# Allow all origins so the GitHub Pages frontend can call this
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class MessageRequest(BaseModel):
    message: str

@app.get("/")
def root():
    return {
        "service": "FraudShield v3",
        "status": "online",
        "classes": CLASS_NAMES,
        "endpoint": "/predict"
    }

@app.get("/health")
def health():
    return {"status": "ok", "device": str(device)}

@app.post("/predict")
def predict_endpoint(req: MessageRequest):
    if not req.message or not req.message.strip():
        return {"error": "Message cannot be empty"}
    return predict(req.message)

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ FastAPI server running on port 8000")

## Step 10 — Cloudflare Tunnel + Copy Button

This creates a public HTTPS endpoint via Cloudflare Tunnel (no account needed).

In [ ]:
import subprocess
import re
import time
from IPython.display import display, HTML

# Download cloudflared binary
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✅ cloudflared downloaded")

# Start the tunnel in background, capture output
cf_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
print("⏳ Waiting for tunnel URL...")
for _ in range(60):
    line = cf_proc.stdout.readline()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if not public_url:
    print("❌ Could not get tunnel URL. Check cloudflared output.")
else:
    display(HTML(f"""
    <div style="
        background: #0f172a;
        border: 1px solid #22d3ee;
        border-radius: 12px;
        padding: 24px 28px;
        margin: 12px 0;
        font-family: 'Segoe UI', sans-serif;
    ">
        <div style="color:#94a3b8; font-size:12px; letter-spacing:2px; text-transform:uppercase; margin-bottom:8px;">🛡️ FraudShield v3 — Public Endpoint</div>
        <div style="display:flex; align-items:center; gap:12px; flex-wrap:wrap;">
            <code id="fs-url" style="
                background:#1e293b;
                color:#22d3ee;
                padding: 10px 16px;
                border-radius: 8px;
                font-size: 15px;
                font-weight: 600;
                flex: 1;
                min-width: 200px;
            ">{public_url}</code>
            <button onclick="
                navigator.clipboard.writeText(document.getElementById('fs-url').innerText);
                this.innerText='✅ Copied!';
                setTimeout(()=>this.innerText='📋 Copy', 2000);
            " style="
                background:#22d3ee;
                color:#0f172a;
                border:none;
                border-radius:8px;
                padding:10px 20px;
                font-size:14px;
                font-weight:700;
                cursor:pointer;
                white-space:nowrap;
            ">📋 Copy</button>
        </div>
        <div style="color:#64748b; font-size:12px; margin-top:10px;">
            POST /predict  ·  GET /health  ·  Keep this tab open
        </div>
    </div>
    """))
    print(f"\nEndpoint: {public_url}")
    print("Paste this URL into the FraudShield frontend on GitHub Pages.")

## Step 11 — Keep Alive (Optional)

Run this cell to print a heartbeat every 5 minutes and prevent the session from going idle.

In [ ]:
import time
from datetime import datetime

print("🟢 Keep-alive running. Interrupt (■) to stop.")
while True:
    print(f"  ♥  {datetime.now().strftime('%H:%M:%S')} — server alive at {public_url}")
    time.sleep(300)